# imports

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
import mlflow
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# config

In [2]:
ASSET = "BTC"
INTERVAL = "4h"
EXPERIMENT = "confidence_threshold"
SOURCE_MODEL = "v4_native"

BEST_PARAMS = {
    'n_estimators': 100,
    'learning_rate': 0.01,
    'max_depth': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
}

BASELINE_V4_ACCURACY = 0.5274   # v4 native 4h tuned accuracy
BASELINE_V3_ACCURACY = 0.5243   # v3 derived 4h

# load data

In [3]:
train_df = pd.read_parquet('../../../data/processed/train_btc_4h_native.parquet')
test_df  = pd.read_parquet('../../../data/processed/test_btc_4h_native.parquet')

y_train = train_df.pop('target_direction')
X_train = train_df.drop(columns=['target_4h'], errors='ignore')

y_test = test_df.pop('target_direction')
X_test = test_df.drop(columns=['target_4h'], errors='ignore')

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   |  y_test:  {y_test.shape}")

X_train: (10579, 33)  |  y_train: (10579,)
X_test:  (2645, 33)   |  y_test:  (2645,)


# mlflow

In [4]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment(f"{ASSET}_{INTERVAL}_{EXPERIMENT}")
mlflow.xgboost.autolog(disable=True)

2026/05/20 11:08:11 INFO mlflow.tracking.fluent: Experiment with name 'BTC_4h_confidence_threshold' does not exist. Creating a new experiment.
